# DBSCAN Sanity Check: Does Cluster Count Ever Decrease as a Video Grows?

**Question**: if we take the same video and look at longer and longer truncations of it, does the DBSCAN cluster count (our "dynamicity" proxy) ever *decrease*? It shouldn't — watching more of a video can only reveal the same or more distinct visual states, never fewer.

**Method**: pick one video that DBSCAN assigned 4 clusters (near the dynamicity threshold) and one assigned 8 clusters (clearly dynamic) from the full-dataset results. For each, build growing sub-clips — last 1s, last 2s, last 3s, ... up to the full video — and re-run the embed + DBSCAN pipeline on each. This mirrors how the model actually consumes a clip: real content is always anchored at the *end* of the window (see `generate_segments` / `CustomVideoDataset` in `videoBased_encoders.py`), so truncation here removes more and more from the *beginning*, not the end.

**Efficiency note**: rather than re-decoding and re-embedding the video once per truncation step, we decode + embed the *full* video once at 5Hz, then for each truncation duration just take the trailing slice of already-computed frame embeddings and re-run DBSCAN on that slice. Much cheaper, same result.

**Assumptions made here that you should double check against the real pipeline**:
- DBSCAN hyperparameters: `eps=25`, `min_samples=2` (the values reported as best in the hyperparameter sweep — ROC-AUC ~0.95, Cohen's d ~2).
- Embedding model: `facebook/dinov2-giant-imagenet1k-1-layer` loaded via `AutoModel` (strips the classification head), CLS token (`last_hidden_state[:, 0, :]`) as the per-frame embedding — matches `frameBased_encoders.py`'s `dino_v2_giant` branch and preprocessing (`do_center_crop=False, do_resize=True, crop_pct=1, size={"shortest_edge": 224}`).
- Frame sampling rate: every 200ms (5Hz), per the documented dynamicity-metric pipeline.
- `dbscan_results.csv` columns: `video` (video ID, matches `{video}.mp4` under the Ego4D video root) and `n_clusters`. If the real column names differ, the inspection cell below will make that obvious immediately.
- This notebook only runs on the cluster (GPU + access to `/braintree/...` paths) — not exercised locally.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from transformers import AutoImageProcessor, AutoModel
from torchcodec.decoders import VideoDecoder

## Config

In [ ]:
DBSCAN_RESULTS_CSV = "/braintree/data2/active/users/aicha/results/entire_dataset/dbscan_analysis_outputs/dbscan_results.csv"
VIDEO_ROOT = "/braintree/data2/active/users/aicha/Ego4D_videos"

MODEL_ID = "facebook/dinov2-giant-imagenet1k-1-layer"
SAMPLING_HZ = 5  # one frame every 200ms, matches the dynamicity-metric pipeline

DBSCAN_EPS = 25
DBSCAN_MIN_SAMPLES = 2

TRUNCATION_STEP_SEC = 1  # grow the trailing window by 1s each step

TARGET_N_CLUSTERS = [4, 8]  # pick one video with each of these cluster counts

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load the full-dataset DBSCAN results and pick two example videos

Inspecting `.columns`/`.head()` first — if `video`/`n_clusters` aren't the real column names, this cell will make that obvious before anything downstream silently does the wrong thing.

In [ ]:
dbscan_df = pd.read_csv(DBSCAN_RESULTS_CSV)
print(dbscan_df.columns.tolist())
dbscan_df.head()

In [ ]:
example_videos = {}
for target in TARGET_N_CLUSTERS:
    matches = dbscan_df.loc[dbscan_df["n_clusters"] == target]
    assert len(matches) > 0, f"No video found with n_clusters == {target}"
    video_id = matches.iloc[0]["video"]
    example_videos[target] = video_id
    print(f"n_clusters={target}  ->  video={video_id}")

example_videos

## Embedding + DBSCAN helpers

In [ ]:
processor = AutoImageProcessor.from_pretrained(
    MODEL_ID,
    use_fast=True,
    do_center_crop=False,
    do_resize=True,
    crop_pct=1,
    size={"shortest_edge": 224},
)
model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()

In [ ]:
@torch.no_grad()
def embed_full_video(video_path, sampling_hz=SAMPLING_HZ):
    """Decode the whole video at `sampling_hz`, return (embeddings, video_duration_sec).

    embeddings: (num_sampled_frames, hidden_size) CLS-token embeddings, in
    chronological order (frame 0 = start of video, last frame = end of video).
    """
    decoder = VideoDecoder(video_path)
    num_frames = decoder.metadata.num_frames
    fps = decoder.metadata.average_fps
    duration_sec = num_frames / fps

    stride = max(1, round(fps / sampling_hz))
    frame_indices = np.arange(0, num_frames, stride)

    frames = decoder.get_frames_at(indices=frame_indices.tolist()).data  # (N, C, H, W), uint8

    inputs = processor(images=list(frames), return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # (N, hidden_size)

    del decoder
    return cls_embeddings, duration_sec, sampling_hz


def cluster_counts(embeddings, eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES):
    """Run DBSCAN on a set of frame embeddings, return (n_clusters, n_noise)."""
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(embeddings)
    n_clusters = len(set(labels) - {-1})
    n_noise = int((labels == -1).sum())
    return n_clusters, n_noise

## Run the sanity check: grow each video from the end, 1s at a time

In [ ]:
results = []

for target_n_clusters, video_id in example_videos.items():
    video_path = os.path.join(VIDEO_ROOT, f"{video_id}.mp4")
    print(f"Embedding {video_path} (original n_clusters={target_n_clusters}) ...")

    embeddings, duration_sec, sampling_hz = embed_full_video(video_path)
    n_frames_total = embeddings.shape[0]

    max_trunc_sec = int(np.floor(duration_sec))
    for trunc_sec in range(TRUNCATION_STEP_SEC, max_trunc_sec + 1, TRUNCATION_STEP_SEC):
        n_frames_kept = min(n_frames_total, round(trunc_sec * sampling_hz))
        if n_frames_kept < DBSCAN_MIN_SAMPLES:
            continue  # DBSCAN needs at least min_samples points to find any cluster

        # keep the LAST n_frames_kept frames -- truncation removes from the
        # beginning, real content stays anchored at the end (matches the
        # model's own padding/segment convention)
        trailing_embeddings = embeddings[-n_frames_kept:]

        n_clusters, n_noise = cluster_counts(trailing_embeddings)
        results.append({
            "video": video_id,
            "original_n_clusters": target_n_clusters,
            "truncation_sec": trunc_sec,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
        })

    # always include the full, untruncated video as the final point
    n_clusters, n_noise = cluster_counts(embeddings)
    results.append({
        "video": video_id,
        "original_n_clusters": target_n_clusters,
        "truncation_sec": duration_sec,
        "n_clusters": n_clusters,
        "n_noise": n_noise,
    })

results_df = pd.DataFrame(results).sort_values(["video", "truncation_sec"]).reset_index(drop=True)
results_df.to_csv("dbscan_sanity_check_results.csv", index=False)
results_df

## Plot: cluster count vs. truncation duration, one panel per video

In [ ]:
fig, axes = plt.subplots(1, len(example_videos), figsize=(7 * len(example_videos), 5), squeeze=False)
axes = axes[0]

for ax, (target_n_clusters, video_id) in zip(axes, example_videos.items()):
    sub = results_df.loc[results_df["video"] == video_id]
    x = np.arange(len(sub))
    width = 0.35

    bars_clusters = ax.bar(x - width / 2, sub["n_clusters"], width, label="Number of clusters")
    bars_noise = ax.bar(x + width / 2, sub["n_noise"], width, label="Number of noise points")

    for bars in (bars_clusters, bars_noise):
        ax.bar_label(bars, padding=2)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{t:.0f}s" for t in sub["truncation_sec"]])
    ax.set_xlabel("Truncation duration (kept from the end of the video)")
    ax.set_ylabel("Count")
    ax.set_title(f"video={video_id}  (full-video n_clusters={target_n_clusters})")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("DBSCAN sanity check: cluster count should never decrease as the video grows")
fig.tight_layout()
fig.savefig("dbscan_sanity_check_plot.png", dpi=150)
plt.show()

## Check monotonicity explicitly

In [ ]:
for video_id, sub in results_df.groupby("video"):
    sub = sub.sort_values("truncation_sec")
    diffs = np.diff(sub["n_clusters"].values)
    n_decreases = int((diffs < 0).sum())
    status = "MONOTONIC (non-decreasing)" if n_decreases == 0 else f"NOT monotonic -- {n_decreases} decrease(s)"
    print(f"video={video_id}: {status}")
    if n_decreases > 0:
        drop_points = sub.iloc[1:][diffs < 0]
        print(drop_points[["truncation_sec", "n_clusters"]])